In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from dotenv import load_dotenv
import os

In [8]:
load_dotenv("../.env")
census_key = os.getenv("CENSUS_API_KEY")

In [9]:
API_KEY    = census_key
STATE_FIPS = "54"                          # West Virginia
YEARS      = [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]

# S1201_C02_001E = Percent never married
# S1201_C05_001E = Percent divorced
VARIABLES  = "S1201_C02_001E,S1201_C03_001E"

In [10]:
# ── STEP 1: Pull ACS 5-Year S1201 for all WV counties ───────────────────────
records = []

for year in YEARS:
    url = (
        f"https://api.census.gov/data/{year}/acs/acs5/subject"
        f"?get=NAME,{VARIABLES}"
        f"&for=county:*"
        f"&in=state:{STATE_FIPS}"
        f"&key={API_KEY}"
    )
    response = requests.get(url)

    # Debug print in case of errors
    print(f"Year: {year} | Status: {response.status_code}")
    if response.status_code != 200:
        print(f"Response: {response.text[:300]}")
        continue

    data = response.json()
    headers = data[0]
    for row in data[1:]:
        record = dict(zip(headers, row))
        record["Year"] = year
        records.append(record)

df = pd.DataFrame(records)
df

Year: 2013 | Status: 200
Year: 2014 | Status: 200
Year: 2015 | Status: 200
Year: 2016 | Status: 200
Year: 2017 | Status: 200
Year: 2018 | Status: 200
Year: 2019 | Status: 200
Year: 2020 | Status: 200
Year: 2021 | Status: 200
Year: 2022 | Status: 200
Year: 2023 | Status: 200


,NAME,S1201_C02_001E,S1201_C03_001E,state,county,Year
0,"Grant County, West Virginia",53.3,7.8,54,023,2013
1,"Summers County, West Virginia",53.2,9.0,54,089,2013
2,"Brooke County, West Virginia",49.5,9.1,54,009,2013
3,"Greenbrier County, West Virginia",52.9,9.2,54,025,2013
4,"Hardy County, West Virginia",46.1,7.7,54,031,2013
...,...,...,...,...,...,...
600,"Webster County, West Virginia",54.3,8.6,54,101,2023
601,"Wetzel County, West Virginia",49.1,9.6,54,103,2023
602,"Wirt County, West Virginia",57.8,10.4,54,105,2023
603,"Wood County, West Virginia",48.6,7.4,54,107,2023


In [11]:
df["FIPS_Code"]        = df["state"] + df["county"]
df["County"]           = df["NAME"].str.replace(", West Virginia", "", regex=False)
df["Pct_Never_Married"] = pd.to_numeric(df["S1201_C02_001E"], errors="coerce")
df["Pct_Divorced"]      = pd.to_numeric(df["S1201_C03_001E"], errors="coerce")

# ── STEP 3: Final structure ───────────────────────────────────────────────────
df = df[["Year", "FIPS_Code", "County", "Pct_Never_Married", "Pct_Divorced"]].copy()
df = df.sort_values(["Year", "FIPS_Code"]).reset_index(drop=True)

df

,Year,FIPS_Code,County,Pct_Never_Married,Pct_Divorced
0,2013,54001,Barbour County,55.1,7.4
1,2013,54003,Berkeley County,53.4,5.6
2,2013,54005,Boone County,56.5,8.3
3,2013,54007,Braxton County,54.3,7.8
4,2013,54009,Brooke County,49.5,9.1
...,...,...,...,...,...
600,2023,54101,Webster County,54.3,8.6
601,2023,54103,Wetzel County,49.1,9.6
602,2023,54105,Wirt County,57.8,10.4
603,2023,54107,Wood County,48.6,7.4


In [12]:
df.to_csv("marital_status.csv", index=False)